In [1]:

import sqlite3
import hashlib
import secrets

# Create database
conn = sqlite3.connect("secure_login.db")
cursor = conn.cursor()

# Create users table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT PRIMARY KEY,
    password_hash TEXT NOT NULL,
    salt TEXT NOT NULL
)
""")
conn.commit()


# Secure password hashing
def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        bytes.fromhex(salt),
        100000
    ).hex()


# Register a user
def register(username, password):
    salt = secrets.token_hex(16)
    password_hash = hash_password(password, salt)

    try:
        cursor.execute(
            "INSERT INTO users VALUES (?, ?, ?)",
            (username, password_hash, salt)
        )
        conn.commit()
        print("Registration successful!")

    except sqlite3.IntegrityError:
        print("Username already exists!")


# Secure login using parameterized SQL
def login(username, password):
    cursor.execute(
        "SELECT password_hash, salt FROM users WHERE username = ?",
        (username,)
    )

    user = cursor.fetchone()

    if user is None:
        print("Login failed!")
        return

    stored_hash, salt = user
    entered_hash = hash_password(password, salt)

    if secrets.compare_digest(stored_hash, entered_hash):
        print("Login successful!")
    else:
        print("Login failed!")


# Test the system
print("SQL INJECTION PREVENTION SYSTEM")

register("student", "MyPassword123")

print("\nCorrect password:")
login("student", "MyPassword123")

print("\nWrong password:")
login("student", "wrongpassword")

print("\nSQL injection test:")
login("' OR '1'='1' --", "anything")

conn.close()


SQL INJECTION PREVENTION SYSTEM
Registration successful!

Correct password:
Login successful!

Wrong password:
Login failed!

SQL injection test:
Login failed!
